## 1. Setup


In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("../../src").resolve()))
from condpred import report
from condpred.experiments import e6

SMOKE = os.environ.get("CONDPRED_SMOKE") == "1"
OUT = Path("../../notebooks/tables/condpred/E6" + ("/smoke" if SMOKE else ""))
FIG = OUT / "figures"
CACHE = "E6/smoke" if SMOKE else "E6"
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)
SPEC = e6.build_initial_spec(SMOKE)
e6.assert_spec_out_dir(SPEC, OUT)


## 2. Write spec


In [ ]:
report.write_spec(SPEC, OUT, modules=e6.E6_MODULES, completion_keys=e6.COMPLETION_KEYS)
e6.verify_spec_immutable(OUT, SPEC, e6.E6_MODULES)


## 3. Leakage test


In [ ]:
rng = np.random.default_rng(0)
leaks = e6.run_leakage_e6(rng)
ldf = pd.DataFrame(leaks)
report.save_table(ldf, OUT, "leakage")
arm_a = ldf.loc[ldf["pipeline"] == "arm_A_lookahead"].iloc[0]
arm_b = ldf.loc[ldf["pipeline"] == "arm_B_causal"].iloc[0]
assert arm_a["verdict"] == "expected" and (not bool(arm_a["pass"])), "arm A lookahead must fail and be labelled expected"
assert arm_b["verdict"] == "expected" and bool(arm_b["pass"]), "arm B must pass"
assert SPEC["primary_test"]["includes_arm_A"] is False
assert not bool(arm_a["in_primary"])


## 4. Data snapshot and quality


In [ ]:
RAW, QREP = e6.load_yahoo_snapshot(SPEC, refresh=False)
SPEC = e6.freeze_panel_tickers(RAW, SPEC)
report.write_spec(
    SPEC, OUT, modules=e6.E6_MODULES,
    completion_keys=e6.COMPLETION_KEYS,
)
report.save_table(QREP, OUT, "data_quality")


## 5. Arm A


In [ ]:
PRICES = e6.close_series(RAW, "BTC-USD")
ARM_A = e6.arm_a_grid(PRICES, SPEC, CACHE)
if len(ARM_A):
    report.save_table(ARM_A, OUT, "armA_grid")


## 6. Arm B


In [ ]:
PACKS = e6.build_packs(
    RAW, SPEC, CACHE, preprocess="window_scale", embedding="primary", include_lrqa=True,
)
OOS = e6.collect_oos(
    PACKS, SPEC, horizon=SPEC["arm_b_horizon_primary"], label="point", models={"DET_LAM": ["DET_LAM"]},
)["DET_LAM"]
ARM_B = e6.summarize_by_asset(OOS, SPEC, seed=SPEC["seed_arm_bc"])
if len(ARM_B):
    report.save_table(ARM_B, OUT, "armB_results")


## 7. Arm C


In [ ]:
ARM_C = e6.summarize_panel(OOS, SPEC, seed=SPEC["seed_arm_bc"] + 1)
if len(ARM_C):
    report.save_table(ARM_C, OUT, "armC_panel")


## 8. Decision


In [ ]:
pywt_ok = e6._pywt_available()
observed = float(ARM_C["panel_delta_auc"].iloc[0]) if ARM_C is not None and len(ARM_C) and "panel_delta_auc" in ARM_C.columns else float("nan")
DIAG = e6.run_diagnostics(PACKS, SPEC, observed)
report.save_table(DIAG["placebo"], OUT, "placebo")
report.save_table(DIAG["within_year"], OUT, "within_year_auc")
DECISION = e6.build_decision(SPEC, ARM_A, ARM_B, ARM_C, pywt_ok=pywt_ok, diagnostics=DIAG)
assert sum(bool(r["in_primary"]) for r in DECISION) == 1
assert all(r["gate"] != "primary" or r["statistic"] == "delta_auc_M1_M0" for r in DECISION)
assert any("placebo" in r["rule"] and "within-year" in r["rule"] for r in DECISION if r["gate"] == "reading")
report.write_decision(DECISION, OUT)
e6.write_assumptions(OUT)


## 9. Secondary


In [ ]:
SECONDARY = e6.run_secondary(RAW, SPEC, PACKS, CACHE)
if len(SECONDARY):
    report.save_table(SECONDARY, OUT, "secondary_holm")


## 10. Figures


In [ ]:
e6.make_figures(ARM_A, ARM_B, FIG, ARM_C)
decision = pd.read_csv(OUT / "decision.csv")
display(decision)
